[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/crystal_waves/crystal_waves.ipynb)

# Elastic Waves in Crystals

This example computes the phase speeds, polarizations, and energy focusing of elastic waves in cubic crystals. A plane wave displaces the material along a polarization direction while its phase advances along a wavevector heading, and crystal anisotropy makes both the wave speed and the direction of energy transport vary with that heading. Where the map from wavevector heading to energy direction folds over itself, an isotropic pulse of acoustic energy focuses onto sharp caustic lines.

A crystal's stiffness is modelled as a map `Plane <- (Plane, Direction, Plane)` from a face, an intrinsic displacement direction, and a wavevector plane to the traction plane on that face. Binding a wave's heading as both the face and the wavevector gives the Christoffel map `crystals(heading, Direction, heading)` from displacements to forces, whose eigenpairs against the displacement metric `Direction.dot(Direction)` give density times squared speed and the three wave polarizations; binding the polarization as the displacement gives the face power form `crystal(Plane, polarization, heading) & polarization`, whose trace recovers the energy flux direction.

In Cartesian tensor-index notation, crystal elasticity is written with a fourth-rank tensor $C_{ijkl}$ subject to index symmetries $C_{ijkl} = C_{jikl} = C_{ijlk} = C_{klij}$, the Christoffel eigenvalue problem takes the contracted matrix form $(\Gamma_{ik} - \rho v^2 \delta_{ik}) p_k = 0$ with $\Gamma_{ik} = C_{ijkl} n_j n_l$, and the group velocity is evaluated component by component as $v^{\,\mathrm{g}}_i = C_{ijkl} p_j p_k n_l / (\rho v)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import VGA3D as ga
from examples.util.animation import save_figure
from examples.mechanics.crystal_waves import render

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Faces, wavevectors and forces are extrinsic planes; displacements, polarizations and energy flux are intrinsic directions.
Plane = ga.gatype.vector()
Direction = ga.gatype.antivector()
# Traction on a face from the face, the displacement and its wavevector.
Stiffness = ga.gatype((Plane, Plane, Direction, Plane))             # Plane+ <- (Plane+, Direction-, Plane+)
# The crystal's cube axes singled out by cubic anisotropy.
axes = mv.basis()                                                   # [3] Plane+
x_plane, y_plane, z_plane = axes                                    # Plane+, Plane+, Plane+
print(Stiffness)

## Cubic crystal stiffness

An isotropic solid contributes two traction terms on a face. Dilation pairs the wavevector with the displacement without using the metric and pushes directly along the face, whereas shear couples through `Direction.dual_inverse()` using the identity `face | (displacement.dual_inverse() ^ wavevector) == (face & displacement) * wavevector - (face | wavevector) * displacement.dual_inverse()` so all three arguments stay in map order. A cubic crystal adds one anisotropic term in which each cube axis reads the face, the displacement, and the wavevector, scaled by the departure from isotropy `c11 - c12 - 2 * c44`.

In Cartesian tensor-index notation, the cubic stiffness tensor is written with Kronecker deltas and a sum of fourfold products over the crystal basis vectors as $C_{ijkl} = c_{12}\,\delta_{ij}\delta_{kl} + c_{44}(\delta_{ik}\delta_{jl} + \delta_{il}\delta_{jk}) + (c_{11} - c_{12} - 2c_{44}) \sum_{a=1}^3 e^{(a)}_i e^{(a)}_j e^{(a)}_k e^{(a)}_l$.

In [ ]:
# Silicon and strongly anisotropic beta-brass: elastic constants in GPa and densities in g/cm^3 (speeds in km/s).
names = ["silicon", "beta-brass"]
n_materials = len(names)
c11 = np.array([165.7, 129.1])
c12 = np.array([63.9, 109.7])
c44 = np.array([79.6, 82.4])
density = np.array([2.33, 7.60])

In [ ]:
def stiffness(c11: np.ndarray, c12: np.ndarray, c44: np.ndarray) -> Stiffness:
    """Cubic crystal stiffness maps (`[...] Plane+ <- (Plane+, Direction-, Plane+)`) from elastic constants `c11`, `c12` and `c44`."""
    # Dilation pushes on the face by `wavevector & displacement`, shear couples through `Direction.dual_inverse()`, and each cube axis reads all three inputs.
    dilation = Plane * (Direction & Plane)                          # Plane+ <- (Plane+, Direction-, Plane+)
    shear = 2 * (Plane & Direction) * Plane - (Plane | (Direction.dual_inverse() ^ Plane))   # Plane+ <- (Plane+, Direction-, Plane+)
    cubic = (axes * (axes | Plane) * (axes & Direction) * (axes | Plane)).sum()   # Plane+ <- (Plane+, Direction-, Plane+)
    # Scalar extensors lift the elastic constants into the context and carry any material batch axis.
    c11, c12, c44 = (mv.scalar(np.asarray(value)[..., None]) for value in (c11, c12, c44))   # [...] Scalar+
    return c12 * dilation + c44 * shear + (c11 - c12 - 2 * c44) * cubic   # [...] Plane+ <- (Plane+, Direction-, Plane+)


crystals = stiffness(c11, c12, c44)                                 # [n_materials] Plane+ <- (Plane+, Direction-, Plane+)
# Normal stretch along x pulls on the x face with c11 and on the y face with c12; shear of the xy face pulls with c44.
x_direction, y_direction = x_plane.dual(), y_plane.dual()           # Direction-, Direction-
sampled_moduli = [
    crystals(x_plane, x_direction, x_plane) & x_direction,
    crystals(y_plane, x_direction, x_plane) & y_direction,
    crystals(y_plane, x_direction, y_plane) & x_direction,
]                                                                   # [n_materials] Scalar+
print(crystals)
print("c11, c12, c44 read back:", [modulus.to_array() for modulus in sampled_moduli])

## Phase speeds and polarizations

For a plane wave of heading `heading`, the wavefront face coincides with the wavevector plane, so `crystal(heading, Direction, heading)` gives the Christoffel map from displacements to restoring tractions. Pairing that traction with a second displacement forms a symmetric bilinear form `christoffel(Direction) & Direction`, whose generalized eigenpairs against `Direction.dot(Direction)` give density times squared phase speed and the three mutually orthogonal polarizations.

In Cartesian tensor-index notation, the Christoffel matrix is assembled as $\Gamma_{ik} = C_{ijkl}\, n_j n_l$ and solved through the characteristic determinant $\det(\Gamma_{ik} - \rho v^2 \delta_{ik}) = 0$.

In [ ]:
# Cube edge [100], face diagonal [110], and body diagonal [111].
symmetry_headings = mv.vector([[1, 0, 0], [1, 1, 0], [1, 1, 1]]).normalized()   # [3] Plane+

In [ ]:
def waves(crystal: Stiffness, heading: Plane) -> tuple[Scalar, Direction]:
    """Wave moduli (`[..., 3] Scalar+`, density times squared speed) and polarizations (`[..., 3] Direction-`) ordered slowest first along `heading`."""
    christoffel = crystal(heading, Direction, heading)              # [...] Plane+ <- Direction-
    return (christoffel(Direction) & Direction).eigh(Direction.dot(Direction))   # [..., 3] Scalar+, [..., 3] Direction-


symmetry_moduli, symmetry_polarizations = waves(crystals[:, None], symmetry_headings)   # [n_materials, 3, 3] Scalar+, [n_materials, 3, 3] Direction-
symmetry_speeds = (symmetry_moduli / density[:, None, None]).square_root()   # [n_materials, 3, 3] Scalar+
# Along a cube edge [100], the closed-form speeds are sqrt(c44 / rho) twice and sqrt(c11 / rho).
edge_speeds = np.sqrt(np.stack([c44, c44, c11], axis=-1) / density[:, None])   # [n_materials, 3]
print("speeds along [100], [110], [111]:\n", symmetry_speeds.to_array())
print("closed-form speeds along [100]:", edge_speeds)

## Energy flux and phonon focusing

Acoustic power through any face comes from the wave's traction paired with the material velocity along `polarization`. Evaluating `crystal(Plane, polarization, heading) & polarization` yields that power as a linear form on faces, and tracing `Direction * power` recovers the intrinsic direction of energy transport; dividing by density times phase speed gives the group velocity. Although pairing the group velocity with `heading` always reproduces the phase speed, crystal anisotropy deflects the group velocity sideways across the wavefront, concentrating uniformly distributed wavevector headings into bright phonon-focusing caustics on a crystal face.

In Cartesian tensor-index notation, the acoustic Poynting flux and group velocity require contracting three indices of $C_{ijkl}$ against the polarization and wave normal as $v^{\,\mathrm{g}}_i = C_{ijkl}\, p_j p_k n_l / (\rho v)$.

In [ ]:
n_headings = 200_000
rng = np.random.default_rng(0)
sphere_headings = mv.vector(rng.normal(size=(n_headings, 3))).normalized()   # [n_headings] Plane+

In [ ]:
def energy_flow(crystal: Stiffness, heading: Plane, polarization: Direction, density: np.ndarray) -> Direction:
    """Group velocities (`[..., 3] Direction-`) of the three wave modes from the face power form divided by density times phase speed."""
    # Broadcast the crystal, its density and the heading across the three wave modes at each heading.
    crystal, density, wave_heading = crystal[..., None], density[..., None], heading[..., None]   # [..., 1] Plane+ <- (Plane+, Direction-, Plane+), [..., 1], [..., 1] Plane+
    wave_modulus = crystal(wave_heading, polarization, wave_heading) & polarization   # [..., 3] Scalar+
    # Traction paired with polarization gives power through each face; tracing against `Direction` recovers the energy flux direction.
    power = crystal(Plane, polarization, wave_heading) & polarization   # [..., 3] Scalar+ <- Plane+
    return (Direction * power).trace(1, 2) / (wave_modulus * density).square_root()   # [..., 3] Direction-


sphere_moduli, sphere_polarizations = waves(crystals[:, None], sphere_headings)   # [n_materials, n_headings, 3] Scalar+, [n_materials, n_headings, 3] Direction-
sphere_velocities = energy_flow(crystals[:, None], sphere_headings, sphere_polarizations, density[:, None])   # [n_materials, n_headings, 3] Direction-
save_figure(render.focusing_figure(sphere_velocities, names), "crystal_waves_focusing");

Bright lines mark caustics where the map from wavevector heading to group-velocity landing folds over itself on the cube face. Silicon's two shear modes focus into a central box and diagonal ridges, whereas strongly anisotropic beta-brass folds its shear energy into broad loops and fourfold petals while the nearly isotropic compressional mode remains diffuse.

## Wave surfaces in a cube face

Sweeping the wavevector heading around a single cube face traces a cross-section of the three group-velocity wave surfaces, turning the focusing folds seen on the detector face into cusps.

In [ ]:
n_angles = 3000
angles = np.linspace(0.0, 2 * np.pi, n_angles, endpoint=False)

In [ ]:
# Turn the x plane through every angle in the z = 0 cube face.
face_rotors = (mv.xy * (-angles / 2)).exp()                         # [n_angles] Rotor+
face_headings = face_rotors >> mv.x                                 # [n_angles] Plane+
face_moduli, face_polarizations = waves(crystals[:, None], face_headings)   # [n_materials, n_angles, 3] Scalar+, [n_materials, n_angles, 3] Direction-
face_velocities = energy_flow(crystals[:, None], face_headings, face_polarizations, density[:, None])   # [n_materials, n_angles, 3] Direction-
save_figure(render.wave_fronts_figure(face_velocities, names), "crystal_waves_fronts");

Red, blue, and purple markers trace the slow shear, fast shear, and compressional wave surfaces after unit travel time along the `[100]` and `[010]` crystal axes. In silicon the slow shear surface develops small cusps along the axes, whereas in beta-brass the shear surfaces fold deeply into wide triangular cusps that generate the caustic loops above.

In [ ]:
# checks
for modulus, expected_constant in zip(sampled_moduli, (c11, c12, c44)):
    assert ((modulus - expected_constant).abs() < 1e-8 * expected_constant).all()
assert ((symmetry_speeds[:, 0] - edge_speeds).abs() < 1e-7 * edge_speeds).all()
# Along every heading on the sphere and in the cube face, the group velocity paired with its heading equals the phase speed.
sphere_speeds = (sphere_moduli / density[:, None, None]).square_root()   # [n_materials, n_headings, 3] Scalar+
face_speeds = (face_moduli / density[:, None, None]).square_root()   # [n_materials, n_angles, 3] Scalar+
assert (((sphere_headings[:, None] & sphere_velocities) - sphere_speeds).abs() < 1e-7 * sphere_speeds).all()
assert (((face_headings[:, None] & face_velocities) - face_speeds).abs() < 1e-7 * face_speeds).all()